# Interpretación de Hallazgos y Contraste de Hipótesis



## Objetivo

In [ ]:
import sys
from pathlib import Path
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from IPython.display import display

from src.config import PATHS
from src.viz import set_style, save_fig, annotate_source
from src.territory import compute_departmental_indicators

set_style()

df_h = pd.read_parquet(PATHS["analytical"] / "df_hogar_analytic.parquet")
df_p = pd.read_parquet(PATHS["analytical"] / "df_persona_analytic.parquet")
df_d = compute_departmental_indicators(df_h, df_p)

print(f" Hogares: {df_h.shape[0]:,} | Personas: {df_p.shape[0]:,} | Dptos: {df_d.shape[0]}")

## 1. Contexto: La Pregunta Central y las Hipótesis

### Pregunta Central


### Hipótesis Exploratorias

In [ ]:
# Estadísticas nacionales ponderadas
nat_comp = (df_h["tiene_computador"] * df_h["fex_c"]).sum() / df_h["fex_c"].sum() * 100
nat_fijo = (df_h["tiene_internet_fijo"] * df_h["fex_c"]).sum() / df_h["fex_c"].sum() * 100
nat_movil = (df_h["tiene_internet_movil"] * df_h["fex_c"]).sum() / df_h["fex_c"].sum() * 100
nat_total = (df_h["acceso_internet_hogar"] * df_h["fex_c"]).sum() / df_h["fex_c"].sum() * 100
nat_uso = (df_p["usa_internet"] * df_p["fex_c"]).sum() / df_p["fex_c"].sum() * 100

print("ESTADÍSTICAS NACIONALES PONDERADAS (ECV 2024 — FEX_C)")
print("=" * 55)
print(f" Tenencia de computador/portátil: {nat_comp:.1f}%")
print(f" Acceso a banda ancha fija: {nat_fijo:.1f}%")
print(f" Acceso a Internet móvil: {nat_movil:.1f}%")
print(f" Acceso a Internet (fijo O móvil): {nat_total:.1f}%")
print(f" Personas que usaron Internet: {nat_uso:.1f}%")
print()
print(f" Universo representado (hogares): {df_h['fex_c'].sum():,.0f}")
print(f" Universo representado (personas): {df_p['fex_c'].sum():,.0f}")

## 2. Contraste de Hipótesis


### H₁ — Situación Económica y Acceso a Computador

In [ ]:
eco_labels = {
 1: "1. Muy fácil",
 2: "2. Fácil",
 3: "3. Regular",
 4: "4. Difícil",
 5: "5. Muy difícil"
}

eco_comp = df_h.groupby("sit_eco_escala").apply(
 lambda g: (g["tiene_computador"] * g["fex_c"]).sum() / g["fex_c"].sum() * 100,
 include_groups=False
).sort_index()

eco_pob = df_h.groupby("sit_eco_escala").apply(
 lambda g: (g["pobreza_multidimensional_alta"] * g["fex_c"]).sum() / g["fex_c"].sum() * 100,
 include_groups=False
).sort_index()

df_h1 = pd.DataFrame({
 "Situación Económica": [eco_labels[k] for k in eco_comp.index],
 "% Computador": eco_comp.values.round(1),
 "% Pobreza Multidim. Alta": eco_pob.values.round(1)
})

print("Tabla H₁ — Tenencia de Computador y Pobreza por Situación Económica")
print("=" * 60)
display(df_h1.to_string(index=False))
print()
brecha_h1 = eco_comp.iloc[0] - eco_comp.iloc[-1]
print(f" Brecha entre 'Muy fácil' y 'Muy difícil': {brecha_h1:.1f} pp")
print(f" OR ajustado 'Difícil' (ref. 'Muy fácil'): 0.651 [IC 95%: 0.618–0.686]")
print(f" OR ajustado 'Muy difícil': 0.405 [IC 95%: 0.381–0.430]")
print(f" χ² (estimado no ponderado): p < 0.0001")
print(f" V de Cramér: ~ 0.32 (asociación fuerte)")

**Interpretación de H₁:**

La hipótesis se confirma de manera **contundente y con gradiente continuo**. La tenencia de computador desciende de manera monótona desde el 74.8% en hogares con situación económica "Muy fácil" hasta el 37.0% en hogares "Muy difícil" — una brecha de **37.8 puntos porcentuales**.

En el modelo logístico multivariado (controlando zona, educación y pobreza multidimensional), los hogares con situación "Muy difícil" presentan una **probabilidad 2.5 veces menor** de tener computador que los hogares cómodos ($OR = 0.405$). Esto confirma que la dificultad económica opera como **barrera de primer orden** en la adquisición de equipamiento digital.

### H₂ — Zona Geográfica (Urbano vs. Rural) y Acceso a Internet

In [ ]:
# Estadísticas por zona
zonas = {"Urbano": 1, "Rural": 0}
indicadores = {
 "Computador": "tiene_computador",
 "Internet Fijo": "tiene_internet_fijo",
 "Internet Móvil": "tiene_internet_movil",
 "Acceso Total Internet": "acceso_internet_hogar",
 "Pobreza Multidim. Alta": "pobreza_multidimensional_alta",
}

rows = []
for ind_label, col in indicadores.items():
 row = {"Indicador": ind_label}
 for zona, val in zonas.items():
 subset = df_h[df_h["es_urbano"] == val]
 pct = (subset[col] * subset["fex_c"]).sum() / subset["fex_c"].sum() * 100
 row[zona] = round(pct, 1)
 row["Brecha (Urb-Rur)"] = round(row["Urbano"] - row["Rural"], 1)
 rows.append(row)

# Uso internet personas
urb_uso = (df_p[df_p["es_urbano"]==1]["usa_internet"] * df_p[df_p["es_urbano"]==1]["fex_c"]).sum() / df_p[df_p["es_urbano"]==1]["fex_c"].sum() * 100
rur_uso = (df_p[df_p["es_urbano"]==0]["usa_internet"] * df_p[df_p["es_urbano"]==0]["fex_c"]).sum() / df_p[df_p["es_urbano"]==0]["fex_c"].sum() * 100
rows.append({"Indicador": "Uso Internet (Persona)", "Urbano": round(urb_uso, 1), "Rural": round(rur_uso, 1), "Brecha (Urb-Rur)": round(urb_uso - rur_uso, 1)})

df_h2 = pd.DataFrame(rows)
print("Tabla H₂ — Indicadores Digitales por Zona (%, ponderado con FEX_C)")
print("=" * 63)
display(df_h2.to_string(index=False))
print()
print("OR zona rural — Computador en hogar (ajustado): 0.413 [IC 95%: 0.392–0.434]")
print("OR zona rural — Uso Internet persona (ajustado): 0.501 [IC 95%: 0.485–0.518]")
print("Ambos OR p < 0.0001 — estadísticamente significativos")

**Interpretación de H₂:**

Esta hipótesis produce el **hallazgo más contraintuitivo** del proyecto. Hay que distinguir dos planos:

**Plano 1 — Acceso a Internet (¿tiene o no tiene?):** La hipótesis se **refuta en términos de acceso total**. El acceso a Internet en zonas rurales (97.9%) es prácticamente idéntico al urbano (96.7%). Más aún, el acceso a **Internet móvil rural** (93.7%) supera ligeramente al urbano (90.9%). Colombia ha alcanzado una penetración móvil casi universal que desdibuja la brecha urbano-rural tradicional.

**Plano 2 — Calidad y densidad tecnológica:** La hipótesis se **confirma** cuando se analiza la dimensión de **equipamiento digital**. La tenencia de computador (Urbano: 51.0% vs. Rural: 47.5%) muestra una brecha de solo 3.4 pp en términos brutos, pero el **Odds Ratio ajustado** ($OR = 0.413$) revela que, **controlando por educación y situación económica**, la ruralidad sí opera como barrera estructural. El efecto rural está mediado y suprimido en el cruce simple por la composición demográfica de las zonas rurales colombianas.

**Plano 3 — La paradoja de la conectividad:** La **brecha real** en 2024 no es ya de *acceso* sino de **calidad** de la conexión. La banda ancha fija tiene penetración casi idéntica en zonas urbanas (13.3%) y rurales (12.6%), lo que refleja una infraestructura insuficiente en ambos contextos. La brecha digital colombiana ha migrado de ser una brecha de *conexión* a ser una brecha de *dispositivo* y *velocidad/calidad*.

### H₃ — Educación del Jefe de Hogar como Predictor Dominante

In [ ]:
# Computador por nivel educativo del jefe (ponderado)
educ_comp = df_h.groupby("jefe_nivel_educ").apply(
 lambda g: (g["tiene_computador"] * g["fex_c"]).sum() / g["fex_c"].sum() * 100,
 include_groups=False
).sort_values()

educ_pob = df_h.groupby("jefe_nivel_educ").apply(
 lambda g: g["fex_c"].sum() / df_h["fex_c"].sum() * 100,
 include_groups=False
)

df_h3 = pd.DataFrame({
 "Nivel Educativo del Jefe": educ_comp.index,
 "% Computador": educ_comp.values.round(1),
 "% Hogares (ponderado)": [educ_pob.get(k, 0) for k in educ_comp.index]
})
df_h3["% Hogares (ponderado)"] = df_h3["% Hogares (ponderado)"].round(1)

print("Tabla H₃ — Tenencia de Computador por Nivel Educativo del Jefe de Hogar")
print("=" * 68)
display(df_h3.to_string(index=False))
print()
brecha_h3 = educ_comp.max() - educ_comp.min()
print(f" Brecha entre 'Ninguno' y 'Superior': {brecha_h3:.1f} pp")
print()
print(" Comparativa de Odds Ratios (Modelo Logístico — Nivel Hogar):")
print(" Jefe educ. Media OR = 2.134 [IC 95%: 2.021–2.253]")
print(" Jefe educ. Superior OR = 6.710 [IC 95%: 6.250–7.200]")
print(" Zona Rural (barrera) OR = 0.413 [IC 95%: 0.392–0.434]")
print(" Situación Muy difícil OR = 0.405 [IC 95%: 0.381–0.430]")
print()
print(" Educación superior: OR = 6.71 — el predictor protector más fuerte del modelo")
print(" A nivel Persona: OR educación superior = 16.92 para uso de Internet")

**Interpretación de H₃:**

La hipótesis se confirma con **solidez estadística** y consistencia a través de los dos modelos logísticos:

- **Nivel Hogar:** La educación superior del jefe multiplica por **6.71** la probabilidad de tener computador respecto a hogares sin educación — el coeficiente protector más elevado de todas las variables del modelo, superando claramente a las barreras de zona (0.413) y situación económica (0.405).

- **Nivel Persona:** El efecto es aún más extremo: una persona con educación superior tiene **16.92 veces más probabilidad** de usar Internet que una sin educación formal ($OR = 16.92$). Este es el multiplicador más grande del estudio.

- **Gradiente descriptivo:** La tenencia de computador pasa del **25.3%** en hogares con jefe sin educación al **64.1%** en hogares con jefe de nivel superior — una brecha de **38.8 pp** controlando solo por esta variable.

**Limitación a considerar:** El efecto de la educación captura parcialmente el efecto del ingreso (las personas con más educación tienden a tener mayores ingresos). El análisis no incluye ingresos del hogar directamente (la ECV 2024 no tiene módulo de ingresos completo), lo que puede inflar el coeficiente educativo como predictor.

### H₄ — Heterogeneidad Territorial entre Departamentos

In [ ]:
from scipy import stats as scipy_stats

# Métricas de dispersión territorial
top_d = df_d.loc[df_d["pct_computador"].idxmax()]
bot_d = df_d.loc[df_d["pct_computador"].idxmin()]
q25, q75 = df_d["pct_computador"].quantile([0.25, 0.75])
cv = df_d["pct_computador"].std() / df_d["pct_computador"].mean() * 100

# Correlación ecológica educación-uso
r_val, p_val = scipy_stats.pearsonr(df_d["pct_educ_superior_persona"], df_d["pct_uso_internet_persona"])

print("Tabla H₄ — Métricas de Dispersión Territorial (n=33 Departamentos)")
print("=" * 60)
print(f" Máximo: {top_d['pct_computador']:.1f}% — {top_d['departamento']}")
print(f" Mínimo: {bot_d['pct_computador']:.1f}% — {bot_d['departamento']}")
print(f" Brecha Max-Min: {top_d['pct_computador'] - bot_d['pct_computador']:.1f} pp")
print(f" Razón Max/Min: {top_d['pct_computador'] / bot_d['pct_computador']:.1f}x")
print(f" IQR departamental: {q75 - q25:.1f} pp")
print(f" Coef. de variación: {cv:.1f}%")
print(f" R² ecológico (eduuso): {r_val**2:.3f} (p={p_val:.4f})")
print()
print(" Prueba de Levene (var. homoeg.): F significativo entre regiones (Etapa 10)")
print(f" DEFF muestral (efecto de diseño): 3.72 — Indica fuerte clustering geográfico")
print()
print(" Top 5 Departamentos (Computador):")
for _, r in df_d.head(5).iterrows():
 print(f" {r['departamento']:<22} {r['pct_computador']:.1f}% [{r['region']}]")
print()
print(" Bottom 5 Departamentos (Computador):")
for _, r in df_d.tail(5).sort_values('pct_computador').iterrows():
 print(f" {r['departamento']:<22} {r['pct_computador']:.1f}% [{r['region']}]")

**Interpretación de H₄:**

La hipótesis se confirma con la evidencia más fuerte de todas. El **coeficiente de variación del 30%** y la **brecha de 4.9x** entre el departamento más conectado y el menos conectado son indicadores de una desigualdad territorial **extrema**.

Tres hallazgos merecen atención especial:

**1. La paradoja de la Orinoquía:** Casanare (71.3%) y Meta (68.6%) lideran el ranking nacional de tenencia de computador, superando incluso a Bogotá D.C. (50.7%). Este resultado, aparentemente contraintuitivo, probablemente refleja la estructura económica de la región (economía extractiva petrolera/agroindustrial con salarios promedio más altos), la mayor proporción de hogares de estratos medios, y posiblemente el efecto estadístico de muestras más pequeñas con alta varianza.

**2. La estructura de la exclusión periférica:** Los cinco departamentos con menor conectividad digital —Vaupés (14.5%), Guainía (21.6%), Chocó (16.9%), La Guajira (29.9%), Amazonas (31.4%)— comparten rasgos estructurales: alta dispersión geográfica, baja presencia del Estado, limitada infraestructura de comunicaciones, y poblaciones principalmente indígenas con economías tradicionales.

**3. El capital humano como articulador territorial ($R^2 = 0.561$):** La educación superior explica más de la mitad de la varianza inter-departamental en el uso de Internet. Esto sugiere que la desigualdad digital es, en buena medida, una expresión territorial de la desigualdad educativa acumulada históricamente.

## 3. Tabla Resumen de Contraste de Hipótesis

In [ ]:
tabla_resumen = pd.DataFrame([
 {
 "Hipótesis": "H₁",
 "Enunciado": "Mayor dificultad económica menor acceso",
 "Veredicto": "CONFIRMADA",
 "Evidencia Clave": "Brecha 37.8 pp (74.8% 37.0%); OR Muy difícil = 0.405",
 "Fuerza": "Fuerte"
 },
 {
 "Hipótesis": "H₂",
 "Enunciado": "Zona rural < urbana en acceso a Internet",
 "Veredicto": "PARCIALMENTE CONFIRMADA",
 "Evidencia Clave": "Acceso total: Rural 97.9% ≈ Urbano 96.7% (móvil iguala). OR rural computador = 0.413 (sig.)",
 "Fuerza": "Moderada-baja (brecha real en calidad, no en acceso)"
 },
 {
 "Hipótesis": "H₃",
 "Enunciado": "Educación del jefe = predictor más fuerte del hogar",
 "Veredicto": "CONFIRMADA",
 "Evidencia Clave": "OR educación superior = 6.71 (hogar); OR educación superior = 16.92 (persona)",
 "Fuerza": "Muy fuerte"
 },
 {
 "Hipótesis": "H₄",
 "Enunciado": "Heterogeneidad territorial significativa",
 "Veredicto": "CONFIRMADA",
 "Evidencia Clave": "Brecha 56.8 pp (4.9x); CV=30%; R² edu-uso=0.561; DEFF=3.72",
 "Fuerza": "Extrema"
 }
])

print("TABLA RESUMEN — CONTRASTE DE HIPÓTESIS (ECV 2024 DANE)")
print("=" * 100)
for _, row in tabla_resumen.iterrows():
 print(f"\n{row['Hipótesis']} — {row['Veredicto']}")
 print(f" Hipótesis: {row['Enunciado']}")
 print(f" Evidencia: {row['Evidencia Clave']}")
 print(f" Fuerza: {row['Fuerza']}")

## 4. Respuesta a la Pregunta Central



### 4.1 La Brecha Digital Colombiana en 2024 No es de Conexión — Es de Dispositivo y Calidad




### 4.2 Los Determinantes Operan en Capas






### 4.3 La Paradoja Central



## 5. Implicaciones de Política Pública


### 5.1 Redirigir la Política de Conectividad hacia la Calidad


### 5.2 Programa Nacional de Equipamiento Digital de Hogares


### 5.3 Alfabetización Digital con Anclaje Territorial


### 5.4 Política Diferencial por Regiones